# Customer Churn Prediction - EDA
### Based on: Matuszelanski & Kopczewska (2022) - Customer Churn in Retail E-Commerce Business: Spatial and Machine Learning Approach
### Journal: Journal of Theoretical and Applied Electronic Commerce Research

In [0]:
from pyspark.sql.functions import col, count, when, datediff, current_date
from pyspark.sql.functions import max as spark_max, date_format, to_date
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

COLOR_MAIN  = "#1B3A6B"
COLOR_WARN = "#E65C00"
COLOR_OK = "#00695C"
COLOR_BAD = "#B71C1C"
COLOR_NEUTRAL = "#9E9E9E"

print("imports done")

In [0]:
orders_df    = spark.table("bharatmart.silver.slv_orders")
sessions_df  = spark.table("bharatmart.silver.slv_sessions")
cart_df      = spark.table("bharatmart.silver.slv_cart_events")
reviews_df   = spark.table("bharatmart.silver.slv_reviews")
refunds_df   = spark.table("bharatmart.silver.slv_refunds")
payments_df  = spark.table("bharatmart.silver.slv_payments")
customers_df = spark.table("bharatmart.silver.slv_customers")

print("all tables loaded")

In [0]:
print(f"orders: {orders_df.count():>10,}")
print(f"sessions: {sessions_df.count():>10,}")
print(f"cart_events: {cart_df.count():>10,}")
print(f"reviews: {reviews_df.count():>10,}")
print(f"refunds: {refunds_df.count():>10,}")
print(f"payments: {payments_df.count():>10,}")
print(f"customers: {customers_df.count():>10,}")

In [0]:
display(orders_df.limit(5))

In [0]:
total_orders = orders_df.count()

In [0]:
null_counts = orders_df.select(
    count(when(col("customer_id").isNull(), 1)).alias("customer_id"),
    count(when(col("order_date").isNull(), 1)).alias("order_date"),
    count(when(col("order_amount").isNull(), 1)).alias("order_amount"),
    count(when(col("total_amount").isNull(), 1)).alias("total_amount"),
    count(when(col("discount_amount").isNull(), 1)).alias("discount_amount"),
    count(when(col("order_status").isNull(), 1)).alias("order_status"),
    count(when(col("channel").isNull(), 1)).alias("channel"),
).toPandas()



In [0]:
null_pct = (null_counts.iloc[0] / total_orders * 100).sort_values(ascending=False)
print(null_pct)

total_amount and channel are 99% NULL - useless from orders table.
we will compute total_amount ourselves as order_amount - discount_amount.
channel we will take from sessions table instead.

customer_id 2% NULL - these are ghost orders, will drop them before training.

everything else is clean, no action needed.

In [0]:
fig, ax = plt.subplots(figsize=(10, 5))

null_pct.plot(kind="barh", ax=ax, color=COLOR_BAD)

ax.set_xlabel("NULL %")
ax.set_title("NULL Rate in Orders Table")
plt.tight_layout()
plt.show()

total_amount and channel are clearly the problem columns - almost 100% NULL.
customer_id has a small 2% null which we will drop.
everything else is clean.

this chart confirms what we found in cell 5.

In [0]:
customer_orders = orders_df.filter(col("customer_id").isNotNull()) \
    .groupBy("customer_id") \
    .agg(spark_max("order_date").alias("last_order_date"),
        count("order_id").alias("total_orders")
    )

customer_orders = customer_orders.withColumn(
    "recency_days", datediff(current_date(), col("last_order_date"))
)

display(customer_orders.limit(5))

each row is one customer.
last_order_date is the most recent order that customer placed.
recency_days is how many days ago that was from today.

customers with high recency_days are the ones who have gone silent.
these are our likely churners.

In [0]:
# customers silent for 90+ days AND have at least 2 orders = churned
customer_orders = customer_orders.withColumn(
    "churn_label",
    when((col("recency_days") > 90) & (col("total_orders") >= 2), 1).otherwise(0)
)

display(customer_orders.limit(5))

churn_label = 1 means the customer has gone silent for 90+ days and has placed 
at least 2 orders. these are our confirmed churners.

churn_label = 0 means customer is still active or is a first time buyer.

we require at least 2 orders because a one-time buyer was never really loyal 
to begin with. churn only makes sense if the customer had a buying pattern 
and then stopped.

this matches the paper by Matuszelanski & Kopczewska (2022) exactly - they 
define churn as behavioral silence, not a cancellation event. in e-commerce 
there is no subscription to cancel, so silence is the only signal we have.

90 days is our threshold - if a customer hasn't ordered in 3 months after 
being active, we call them churned.

In [0]:
churn_count = customer_orders.filter(col("churn_label") == 1).count()
active_count = customer_orders.filter(col("churn_label") == 0).count()
total = customer_orders.count()

print(f"churned : {churn_count:,} ({churn_count/total*100:.1f}%)")
print(f"active : {active_count:,} ({active_count/total*100:.1f}%)")
print(f"scale_pos_weight for XGBoost = {active_count/churn_count:.1f}")

27.5% of customers have churned - meaning roughly 1 in 4 customers went silent 
after being active.

this is actually a moderate imbalance. not extreme like 1% churn which some 
telecom datasets have. but still imbalanced enough that we cannot ignore it.

scale_pos_weight = 2.6 means for every churned customer there are 2.6 active 
ones. we will pass this directly to XGBoost so it gives more weight to churned 
customers during training - otherwise the model will just predict everyone as 
active and still get 72% accuracy, which is useless.

the paper used upsampling on the minority class to handle imbalance. we are 
using scale_pos_weight instead - same goal, cleaner approach in XGBoost.

In [0]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(["Active (0)", "Churned (1)"], [active_count, churn_count],
       color=[COLOR_OK, COLOR_BAD], edgecolor="white", width=0.5)

ax.set_ylabel("Customer Count")
ax.set_title("Churn vs Active Customers")
plt.tight_layout()
plt.show()

the bar chart visually confirms the imbalance.
active customers are roughly 2.6x more than churned ones.

this is important because if we train without handling this imbalance, 
the model will be biased towards predicting active. it will look accurate 
but will completely miss churned customers - which defeats the whole purpose.

scale_pos_weight = 2.6 fixes this in XGBoost training.

In [0]:
recency_pdf = customer_orders.select("recency_days").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(recency_pdf["recency_days"].dropna(), bins=50, color=COLOR_MAIN, edgecolor="white")
ax.axvline(x=90, color=COLOR_BAD, linestyle="--", linewidth=2, label="churn threshold 90d")
ax.set_xlabel("Days Since Last Order")
ax.set_ylabel("Customer Count")
ax.set_title("Recency Days Distribution")
ax.legend()
plt.tight_layout()
plt.show()

most customers are concentrated on the left side - meaning they ordered recently.
the long right tail shows customers who have been silent for 500, 1000, even 2000+ days.
those are the deeply churned customers.

the red line at 90 days is our churn threshold.
everyone to the right of that line with 2+ orders is labeled as churned.

the distribution is heavily right skewed - this is normal for e-commerce.
most customers are active, a smaller group has gone completely silent.

recency_days is the single most important feature in churn prediction.
the paper confirms this - behavioral features like recency account for 
60-70% of the model's predictive power.

In [0]:
orders_pdf = customer_orders.select("total_orders").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(orders_pdf["total_orders"].dropna(), bins=50, color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Total Orders per Customer")
ax.set_ylabel("Customer Count")
ax.set_title("Total Orders Distribution")
plt.tight_layout()
plt.show()

most customers have placed between 1 and 20 orders.
the spike at 19-20 is interesting - could be a data pattern or loyalty program 
milestone worth investigating later.
beyond 30 orders the count drops sharply - these are the most loyal customers.

the distribution is right skewed - most customers are low frequency buyers.
a small group of high frequency buyers exists on the right tail.

frequency is the F in RFM - one of the core three features the paper identifies 
as the foundation of churn prediction. low frequency customers are naturally 
at higher churn risk.

we will clip outliers at p99 during feature engineering to avoid the model 
being influenced by extreme values.

In [0]:
amount_pdf = orders_df.filter(col("order_amount") > 0).select("order_amount").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(amount_pdf["order_amount"].dropna(), bins=60, color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Order Amount (Rs.)")
ax.set_ylabel("Order Count")
ax.set_title("Order Amount Distribution")
plt.tight_layout()
plt.show()

the chart is almost unreadable - all orders are squished into the first bar.
this means there are extreme outliers pushing the x-axis to 700,000 Rs.
these are likely bulk or B2B orders - not typical customer behavior.

we need to clip at p99 to see the actual distribution of normal orders.

**Order Amount Distribution**

In [0]:
p99 = float(np.percentile(amount_pdf["order_amount"].dropna(), 99))
print(f"p99 order amount = Rs. {p99:,.0f}")

clipped = amount_pdf[amount_pdf["order_amount"] <= p99]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(clipped["order_amount"], bins=60, color=COLOR_OK, edgecolor="white")
ax.set_xlabel("Order Amount (Rs.) — clipped at p99")
ax.set_ylabel("Order Count")
ax.set_title(f"Order Amount Distribution — clipped at p99 (Rs. {p99:,.0f})")
plt.tight_layout()
plt.show()

p99 means the 99th percentile.
p99 is Rs. 1,789 - meaning 99% of all orders are below this value.
the remaining 1% were extreme outliers going up to Rs. 700,000 which were 
hiding the real distribution.

now we can see clearly - most orders are between Rs. 50 and Rs. 300.
this is typical Indian e-commerce behavior - small to medium ticket purchases.
the distribution is right skewed which is expected.

we will clip total_spent at Rs. 1,789 during feature engineering.
this is the monetary value - the M in RFM.
the paper identifies monetary value as one of the top churn predictors.

**Payment Method Distribution**

In [0]:
pay_pdf = payments_df.groupBy("payment_method").count() \
    .orderBy(col("count").desc()).toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(pay_pdf["payment_method"], pay_pdf["count"], color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Count")
ax.set_title("Payment Method Distribution")
plt.tight_layout()
plt.show()

UPI dominates completely - more than half of all payments are via UPI.
this makes total sense for Indian e-commerce in 2020-2025.
credit card is second, COD third.

net_banking and wallet are minority payment methods.

payment method is a behavioral feature we will use in the model.
customers who pay via COD tend to have different loyalty patterns compared 
to UPI or credit card users - COD customers have a higher return and 
cancellation rate which may signal higher churn risk.

we will encode payment_method as a categorical feature during feature engineering.

**Session Channel Distribution**

In [0]:
chan_pdf = sessions_df.groupBy("channel").count() \
    .orderBy(col("count").desc()).toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(chan_pdf["channel"], chan_pdf["count"], color=COLOR_OK, edgecolor="white")
ax.set_xlabel("Count")
ax.set_title("Session Channel Distribution")
plt.tight_layout()
plt.show()

organic_search is the dominant channel - customers finding BharatMart 
through google search naturally.
social_media and paid_search are second and third.

web_desktop, app_ios, web_mobile, app_android are near zero - these are 
device type channels and have very low session counts.

channel preference is a behavioral signal for churn.
customers who come via push_notification or email are already being 
re-engaged by marketing - they may have higher churn risk.
customers who come via organic_search are self-motivated buyers - 
generally more loyal.

we will use channel as a feature in the model.
since orders.channel was 99% NULL we are correctly taking it from 
sessions table instead.

**Device Type Distribution**

In [0]:
dev_pdf = sessions_df.groupBy("device_type").count() \
    .orderBy(col("count").desc()).toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(dev_pdf["device_type"], dev_pdf["count"], color=COLOR_WARN, edgecolor="white")
ax.set_xlabel("Count")
ax.set_title("Device Type Distribution")
plt.tight_layout()
plt.show()

mobile dominates heavily - almost 1.4 million sessions from mobile.
desktop is second at around 600k.
tablet is small, ios and android are near zero.

this is very typical for Indian e-commerce - mobile first country.
most customers browse and buy on their phones.

device type tells us about customer engagement style.
mobile-only customers who suddenly stop their mobile sessions 
are a strong churn signal.

we will use device_type as a feature in the model.

**Review Rating Distribution**

In [0]:
rating_pdf = reviews_df.groupBy("rating").count().orderBy("rating").toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rating_pdf["rating"].astype(str), rating_pdf["count"],
       color=COLOR_MAIN, edgecolor="white", width=0.6)
ax.set_xlabel("Rating (1-5)")
ax.set_ylabel("Review Count")
ax.set_title("Review Rating Distribution")
plt.tight_layout()
plt.show()

rating 5 is the most common - majority of customers are satisfied.
rating 4 is second. so overall BharatMart has a positive review profile.

rating 1 is surprisingly high - about 50,000 one-star reviews.
rating 2 is the lowest which is a bit unusual. 
customers tend to either love or hate - they go to 1 or 5, not 2.

this is exactly what the paper found on the Olist Brazilian dataset too -
negative review polarization where customers give 1 more than 2.

important finding from the paper - raw rating score alone is a weak 
churn predictor. a customer who gives 1 star is almost as likely to 
reorder as someo

**Cart Events - Action Distribution**

In [0]:
cart_pdf = cart_df.groupBy("action").count() \
    .orderBy(col("count").desc()).toPandas()

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(cart_pdf["action"], cart_pdf["count"], color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Count")
ax.set_title("Cart Events — Action Types")
plt.tight_layout()
plt.show()

cart_add is the highest action - 1.8 million times customers added to cart.
cart_view is second at 1.6 million.
cart_remove is 750k - a lot of items getting removed after being added.

wishlist_add is around 450k - customers saving for later but not buying.
add, remove, update_qty, move_to_wishlist, save_for_later are all very small.

the key insight here is cart_add vs cart_remove ratio.
a customer who adds a lot but removes frequently is showing hesitation behavior.
this is a strong churn signal.

we will compute cart_abandonment_rate as a feature -
(cart_add - purchase) / cart_add per customer.
high abandonment rate means the customer is browsing but not committing -
a classic early warning sign of churn.

**Cart Abandonment Rate**

In [0]:
cart_actions = cart_df.groupBy("customer_id", "action").count()

cart_adds = cart_actions.filter(col("action") == "cart_add") \
    .select("customer_id", col("count").alias("adds"))

purchases = cart_actions.filter(col("action") == "purchase") \
    .select("customer_id", col("count").alias("purchases"))

cart_summary = cart_adds.join(purchases, "customer_id", "left").fillna(0, subset=["purchases"])
cart_summary = cart_summary.withColumn(
    "abandon_rate", (col("adds") - col("purchases")) / col("adds")
)

display(cart_summary.limit(5))

each row is one customer with their cart adds, purchases and abandonment rate.
abandon_rate of 1.0 means the customer added to cart but never purchased at all.
abandon_rate of 0.0 means every cart add resulted in a purchase.

most customers will be somewhere in between.
this is one of the most powerful behavioral features for churn prediction.
a customer who used to buy but now only adds to cart without purchasing 
is clearly losing intent - high churn risk.

**Cart Abandonment Rate Distribution**

In [0]:
abandon_pdf = cart_summary.select("abandon_rate").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(abandon_pdf["abandon_rate"].dropna(), bins=50, color=COLOR_BAD, edgecolor="white")
ax.set_xlabel("Cart Abandonment Rate")
ax.set_ylabel("Customer Count")
ax.set_title("Cart Abandonment Rate Distribution")
plt.tight_layout()
plt.show()

the chart shows almost everyone at abandon_rate = 1.0 - this is wrong.
the reason is cart_events table does not have a "purchase" action.
purchases are recorded in slv_orders table, not in cart events.

we need to fix the abandonment rate - use order count from slv_orders 
as the purchase signal instead of cart_df purchase action.

the paper by Matuszelanski & Kopczewska (2022) specifically highlights 
cart and purchase behavior as key behavioral features for churn prediction.
getting this feature wrong would directly hurt model performance since 
behavioral features account for 60-70% of predictive power.

**Fix Cart Abandonment Rate**

In [0]:
# purchases come from orders table, not cart_events
customer_order_counts = orders_df.filter(col("customer_id").isNotNull()) \
    .groupBy("customer_id") \
    .agg(count("order_id").alias("purchases"))

cart_summary2 = cart_adds.join(customer_order_counts, "customer_id", "left") \
    .fillna(0, subset=["purchases"])

cart_summary2 = cart_summary2.withColumn(
    "abandon_rate", (col("adds") - col("purchases")) / col("adds")
)

display(cart_summary2.limit(5))

now the abandonment rate makes more sense.
a customer with 8 cart adds and 5 purchases has abandon_rate of 0.375 - 
meaning 37.5% of their cart adds did not convert to orders.

this is a realistic behavioral signal now.
previously we were comparing cart adds against a purchase action that 
didnt exist in cart_events - that was giving everyone 1.0 wrongly.

the paper by Matuszelanski & Kopczewska (2022) treats cart and purchase 
behavior as core behavioral features. a rising abandonment rate over time 
is one of the earliest signals that a customer is losing interest - 
which directly links to churn risk.

we will use this corrected abandon_rate in feature engineering.

**Cart Abandonment Rate Distribution - Fixed**

In [0]:
abandon_pdf2 = cart_summary2.select("abandon_rate").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(abandon_pdf2["abandon_rate"].dropna(), bins=50, color=COLOR_BAD, edgecolor="white")
ax.set_xlabel("Cart Abandonment Rate")
ax.set_ylabel("Customer Count")
ax.set_title("Cart Abandonment Rate Distribution — Fixed")
plt.tight_layout()
plt.show()

we are seeing negative abandonment rates - this means some customers 
placed more orders than cart adds.
this happens because not every order goes through the cart - 
some are direct buys, repeat orders, or mobile quick buys.

abandon_rate should only be between 0 and 1.
we need to clip it - anything below 0 becomes 0.

the paper by Matuszelanski & Kopczewska (2022) treats cart behavior 
as a key behavioral feature. they specifically note that behavioral 
features must be correctly computed before training - garbage in 
garbage out applies directly here. a wrong abandon_rate would mislead 
the model about customer intent.

**Fix Negative Abandonment Rates**

In [0]:
from pyspark.sql.functions import least, greatest, lit

cart_summary3 = cart_summary2.withColumn(
    "abandon_rate",
    greatest(lit(0.0), least(lit(1.0), col("abandon_rate")))
)

abandon_pdf3 = cart_summary3.select("abandon_rate").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(abandon_pdf3["abandon_rate"].dropna(), bins=50, color=COLOR_BAD, edgecolor="white")
ax.set_xlabel("Cart Abandonment Rate (0 to 1)")
ax.set_ylabel("Customer Count")
ax.set_title("Cart Abandonment Rate — Clipped 0 to 1")
plt.tight_layout()
plt.show()

now the distribution looks correct and meaningful.
the large spike at 0.0 means many customers purchased everything they added 
to cart - these are highly committed buyers, low churn risk.

the rest of the customers spread between 0.1 and 0.6 abandonment rate.
very few customers go above 0.6 - meaning most people who add to cart 
do eventually buy something.

this is now a clean usable feature for the model.
customers at 0.0 abandon rate are your most loyal - they always follow 
through on their intent.
customers drifting towards 0.4-0.6 are showing hesitation - 
potential churn candidates.

the paper by Matuszelanski & Kopczewska (2022) confirms that behavioral 
features like this - capturing how a customer interacts with the platform 
beyond just orders - significantly improve churn prediction accuracy 
on top of basic RFM features.

**Customer Coverage Across Tables**

In [0]:
total_custs = orders_df.filter(col("customer_id").isNotNull()) \
    .select("customer_id").distinct().count()

sess_custs = sessions_df.select("customer_id").distinct().count()
cart_custs = cart_df.select("customer_id").distinct().count()
review_custs = reviews_df.select("customer_id").distinct().count()
refund_custs = refunds_df.select("customer_id").distinct().count()

print(f"orders : {total_custs:,}  (100%)")
print(f"sessions : {sess_custs:,}  ({sess_custs/total_custs*100:.1f}%)")
print(f"cart : {cart_custs:,}  ({cart_custs/total_custs*100:.1f}%)")
print(f"reviews : {review_custs:,}  ({review_custs/total_custs*100:.1f}%)")
print(f"refunds : {refund_custs:,}  ({refund_custs/total_custs*100:.1f}%)")

coverage is excellent across all tables.
sessions covers 99.9% of customers - almost everyone has browsing data.
cart covers 100% - every customer has cart activity.
reviews covers 92.6% - most customers left a review.
refunds covers 60.6% - more than half of customers have had at least one refund.

this is great news for feature engineering.
we have very few customers with missing behavioral data.
the small gaps will be filled with 0 - a customer with no refunds 
simply gets refund_rate = 0, not a dropped row.

this directly addresses what the paper calls "feature coverage" -
Matuszelanski & Kopczewska (2022) note that behavioral features only 
add value if enough customers have that data. 
at 92%+ coverage across all tables, our behavioral features are solid.

**Monthly Order Volume Over Time**

In [0]:
monthly = orders_df.filter(col("order_date").isNotNull()) \
    .withColumn("ym", date_format(to_date(col("order_date")), "yyyy-MM")) \
    .groupBy("ym").count().orderBy("ym").toPandas()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(len(monthly)), monthly["count"], color=COLOR_MAIN, linewidth=1.5)
ax.fill_between(range(len(monthly)), monthly["count"], alpha=0.15, color=COLOR_MAIN)

tick_pos = list(range(0, len(monthly), 6))
ax.set_xticks(tick_pos)
ax.set_xticklabels([monthly["ym"].iloc[i] for i in tick_pos], rotation=45, ha="right")
ax.set_ylabel("Order Count")
ax.set_title("Monthly Order Volume Over Time")
plt.tight_layout()
plt.show()

the chart shows 6 years of order data from 2020 to 2026.

2020 dip - visible drop around mid 2020, this is the covid lockdown impact.
customers stopped ordering during lockdown period.

festive spikes - clear sharp peaks every year around october-november.
these are diwali and big billion day sales. every year without fail.
the biggest spike is late 2025 going above 70,000 orders in one month.

overall trend - the baseline order volume is clearly growing year on year.
BharatMart is a growing platform, not a declining one.

this time series context is important for our churn model.
a customer who was active during festive season but went silent after 
is very different from one who went silent mid-year.
recency_days captures this - but the model will also pick up on 
seasonal patterns through order frequency features.

the paper by Matuszelanski & Kopczewska (2022) was built on 2016-2018 
Olist data which had no such long time range or festive seasonality. 
our 6-year Indian dataset with festive

**Recency Days - Churned vs Active**

In [0]:
recency_churn = customer_orders.select("recency_days", "churn_label").toPandas()

churned = recency_churn[recency_churn["churn_label"] == 1]["recency_days"]
active  = recency_churn[recency_churn["churn_label"] == 0]["recency_days"]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(active,  bins=50, color=COLOR_OK,  alpha=0.6, label="active",  edgecolor="white")
ax.hist(churned, bins=50, color=COLOR_BAD, alpha=0.6, label="churned", edgecolor="white")
ax.axvline(x=90, color="black", linestyle="--", linewidth=1.5, label="90d threshold")
ax.set_xlabel("Days Since Last Order")
ax.set_ylabel("Customer Count")
ax.set_title("Recency Days — Churned vs Active")
ax.legend()
plt.tight_layout()
plt.show()

this is the most important chart in the entire EDA.

active customers (green) are heavily concentrated on the left - 
most ordered within the last 0-90 days. clean separation.

churned customers (red) start appearing after the 90 day threshold 
and spread all the way to 2000+ days. some customers have been 
silent for 5+ years.

the 90 day threshold line is doing exactly what we designed it to do -
it cleanly separates the two groups. active customers are almost 
entirely to the left, churned customers dominate to the right.

this visual separation tells us recency_days alone will be a very 
strong predictor. the model will heavily rely on this feature.

this perfectly validates what Matuszelanski & Kopczewska (2022) found -
recency is the single most powerful behavioral feature in churn prediction.
it accounts for the largest share of that 60-70% behavioral feature 
importance the paper reports.

**Refund Count Distribution**

In [0]:
refund_counts = refunds_df.groupBy("customer_id") \
    .agg(count("refund_id").alias("refund_count"))

refund_pdf = refund_counts.select("refund_count").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(refund_pdf["refund_count"].dropna(), bins=30, color=COLOR_WARN, edgecolor="white")
ax.set_xlabel("Refund Count per Customer")
ax.set_ylabel("Customer Count")
ax.set_title("Refund Count Distribution")
plt.tight_layout()
plt.show()

most customers who have refunds have only 1-2 refunds total.
the count drops sharply after 2 refunds.
very few customers go above 5 refunds - those are extreme cases.

this is a healthy distribution - refunds are not rampant.
but even 1-2 refunds per customer is a meaningful signal.

a customer who refunds frequently is expressing dissatisfaction -
even if they dont leave a bad review.
this is a behavioral signal that standard RFM misses completely.

the paper by Matuszelanski & Kopczewska (2022) showed that perception 
features like satisfaction signals improve churn prediction on top of 
basic RFM. refund behavior is a stronger and more direct satisfaction 
signal than review ratings because it involves actual money - 
a customer only requests a refund when they are genuinely unhappy.

we will compute refund_rate = refund_count / total_orders per customer
and use it as a feature in the model.

**Session Depth - Pages Viewed Distribution**

In [0]:
pages_pdf = sessions_df.select("pages_viewed").filter(col("pages_viewed").isNotNull()).toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pages_pdf["pages_viewed"].dropna(), bins=50, color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Pages Viewed per Session")
ax.set_ylabel("Session Count")
ax.set_title("Pages Viewed Distribution")
plt.tight_layout()
plt.show()

pages viewed per session ranges from 1 to 20.
there are two clear groups in this distribution.

first group - 1 to 10 pages - high session counts, these are normal browsing sessions.
peak is at 5 pages which is typical casual browsing behavior.

second group - 11 to 20 pages - much lower and flat counts.
these are deep browsing sessions - customers who are actively researching products.

the drop after 10 pages is sharp - most customers dont go beyond 10 pages per session.

think of it like this -
imagine a customer who loves shopping on BharatMart.
every time they open the app they browse through 8-10 pages.
they check shoes, then electronics, then home decor.
they are engaged and interested.

now 2 months later the same customer opens the app,
checks 1-2 pages and closes it.
they are losing interest but have not stopped ordering yet.

this declining engagement happens BEFORE they actually churn.
it is an early warning sign - like a patient whose vitals are
dropping before they actually fall sick.

by the time recency_days crosses 90 days the customer has already churned.
but avg_pages_viewed starts dropping weeks before that.
so it gives us earlier warning than recency alone.

this is why the paper by Matuszelanski & Kopczewska (2022) emphasizes 
using multiple behavioral features together - not just recency. 
each feature catches a different stage of the churn journey.
recency catches it late. session depth catches it early.
together they make the model much more powerful.

we will use avg_pages_viewed per customer as a feature in the model.

**Session Duration Distribution**

In [0]:
duration_pdf = sessions_df.select("session_duration_seconds") \
    .filter(col("session_duration_seconds").isNotNull()).toPandas()

p99_dur = float(np.percentile(duration_pdf["session_duration_seconds"].dropna(), 99))

clipped_dur = duration_pdf[duration_pdf["session_duration_seconds"] <= p99_dur]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(clipped_dur["session_duration_seconds"], bins=50, color=COLOR_MAIN, edgecolor="white")
ax.set_xlabel("Session Duration (seconds) — clipped at p99")
ax.set_ylabel("Session Count")
ax.set_title(f"Session Duration Distribution — clipped at p99 ({p99_dur:,.0f}s)")
plt.tight_layout()
plt.show()

p99 means the 99th percentile.

session duration is clipped at p99 which is 1,718 seconds - about 28 minutes.
so we are looking at normal browsing sessions here, extreme outliers removed.

there are two clear groups again.

first group - 0 to 300 seconds (0 to 5 minutes) - very high session counts.
these are quick browsing sessions. customer opens app, checks something specific, leaves.
peak is around 100-250 seconds which is 2-4 minutes - typical mobile browsing.

second group - 300 to 900 seconds (5 to 15 minutes) - moderate flat counts.
these are longer engaged sessions. customer is actively comparing products,
reading descriptions, checking reviews before deciding.

after 900 seconds the count drops sharply - sessions beyond 15 minutes are rare.

think of it like this -
a loyal engaged customer typically spends 5-15 minutes per session.
they are browsing, comparing, adding to cart, purchasing.

a disengaging customer starts having shorter and shorter sessions.
they open the app, find nothing interesting, close it in under a minute.
session duration dropping from 8 minutes to 1 minute is a silent churn signal -
the customer is still visiting but losing interest fast.

combined with pages_viewed this gives us a complete picture of session engagement.
short duration + few pages = customer is losing interest.

the paper by Matuszelanski & Kopczewska (2022) confirms that behavioral 
engagement features like session depth capture churn signals that pure 
transaction data like RFM completely misses.
we will use avg_session_duration per customer as a feature in the model.

**EDA Summary - ML Cleaning Checklist**

In [0]:
spw = round(active_count / churn_count, 1)

print("=" * 55)
print("ML CLEANING CHECKLIST — Model 1_feature_engineering")
print("=" * 55)
print(f"\nchurn rate : {churn_count/total*100:.1f}%")
print(f"scale_pos_weight : {spw}")
print("\nDROP:")
print("orders where customer_id is NULL")
print("customers with total_orders < 2")
print("\nRECOMPUTE:")
print("total_amount = order_amount - discount_amount")
print("channel = from sessions table not orders")
print("abandon_rate = clip between 0 and 1")
print("abandon_rate = use orders count not cart purchase action")
print("\nFILL NULLS with 0:")
print("session features -> 0")
print("refund_rate -> 0.0")
print("abandon_rate -> 0.0")
print("avg_rating -> 3.0")
print("avg_pages_viewed -> 0")
print("avg_session_dur -> 0")
print("\nCLIP at p99:")
print("order_amount, total_orders")
print("session_duration_seconds")
print("\nFEATURES CONFIRMED FROM EDA:")
print("recency_days, total_orders, total_spent")
print("refund_rate, abandon_rate")
print("avg_pages_viewed, avg_session_duration")
print("payment_method, channel, device_type")
print("=" * 55)

this checklist is our contract before we touch the model.

everything we discovered in EDA is captured here.
no surprises should hit us during feature engineering.

key decisions made:
- 27.5% churn rate - manageable imbalance with scale_pos_weight = 2.6
- total_amount recomputed - saves our monetary feature
- channel taken from sessions - saves our channel feature
- abandon_rate fixed twice - correct source and correct range
- all nulls filled not dropped - absence of activity is itself a signal

9 confirmed features going into feature_engineering:
- recency_days, total_orders, total_spent → RFM core (paper: 60-70% of power)
- refund_rate, abandon_rate → dissatisfaction signals
- avg_pages_viewed, avg_session_duration → engagement depth signals
- payment_method, channel, device_type → behavioral preference signals

the paper by Matuszelanski & Kopczewska (2022) used behavioral, 
perception and location features. we are using behavioral and 
perception features - and our behavioral feature set is richer 
than what the paper had available on the Olist dataset.